# Building Agentic AI Applications
### AI Frontier - Day 10 - Mr. K. Akshay Kumar, Kovan Labs

We'll assemble a mind, one part at a time, and end up with **Wanderlust AI** - a travel agent that plans a real trip from **Coimbatore to Coorg**.

**Brain** -> **Senses** -> **Hands** -> **Will** -> **Team** -> **Framework**.

Copy each snippet as it unlocks, run it (Shift+Enter), and watch the mind grow.

## Snippet 0 - Warm up the engine

_Setup_

Run this first. Installs the Google GenAI SDK (light - no restart needed) and connects with your key.

**Get your free key:** open [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey), create a key. Then in Colab's left sidebar click the **🔑 Secrets** icon, add a secret named `GOOGLE_API_KEY`, paste your key, and toggle **Notebook access** ON.

In [1]:
# Install the Google GenAI SDK (light - no restart needed)
!pip install -q -U google-genai

import json, time, asyncio
from google import genai
from google.genai import types
from google.colab import userdata

# --- API KEY (from the Secrets tab) ---
try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = input("Paste your Gemini API key: ")

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL_ID = "gemini-3.1-flash-lite"

# Smoke test - if this prints, you're live.
print(client.models.generate_content(
    model=MODEL_ID,
    contents="Reply with exactly: Wanderlust AI is online!"
).text)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 822.5/822.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.7.0 which is incompatible.
Wanderlust AI is online!


## Snippet 1 - The Brain - the naive prompt

_1 - The Brain_

An LLM is a brain in a jar. Ask it plainly and you get a plain, guidebook answer.

In [2]:
# The same question, two very different brains. First: no guidance.
prompt = "Plan a 1-day trip to Coorg."

print(f"👤 USER: {prompt}\n" + "="*50)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt + " Keep it under 200 words."
)
print(f"🤖 NAIVE MODEL:\n{response.text}")
# Accurate, but generic - the same 'Raja's Seat, Abbey Falls' answer any
# guidebook gives. This is NOT Wanderlust yet.

👤 USER: Plan a 1-day trip to Coorg.
🤖 NAIVE MODEL:
To maximize a single day in Coorg (Madikeri), start early to beat the crowds.

**Morning (8:00 AM – 11:00 AM):** 
Head straight to **Talakaveri**, the source of the river Kaveri. The hilltop views are breathtaking. On your way back, visit the **Bhagamandala Temple**, a serene site where three rivers converge.

**Midday (12:00 PM – 2:30 PM):** 
Drive to **Abbey Falls**. The lush greenery and the roar of the waterfall are iconic. Afterward, enjoy a traditional Kodava lunch—don't miss the *Pandi Curry* (pork curry) and *Kadambuttu* (rice dumplings) at a local restaurant in Madikeri town.

**Afternoon (3:00 PM – 5:00 PM):** 
Visit **Raja’s Seat**, a stunning garden offering panoramic views of the misty valley. It was once the favorite spot of the Kodagu kings. If time permits, quickly walk through the nearby **Madikeri Fort**.

**Evening (5:30 PM onwards):** 
End your day by picking up authentic Coorg coffee, black pepper, and homemade cho

## Snippet 2 - The Brain - wake up a persona

_1 - The Brain_

Same model, same question - but a persona plus hard constraints gives it a soul. This is the prompt engineering you learned on Day 4, in code.

In [3]:
# Now we wake up a SPECIFIC brain: a persona + hard constraints.
wanderlust_persona = """
ROLE: You are 'Wanderlust AI', a born-and-raised Kodava (Coorg local) guide
who rolls his eyes at tourist traps.
EXPERTISE: Coffee estates, hidden viewpoints, and proper Kodava food
(pandi curry, kadambuttu, akki roti).
TONE: Hip, brief, opinionated.

CONSTRAINTS:
1. BAN the obvious tourist cliches (no generic 'Raja's Seat' filler).
2. Suggest EXACTLY 3 stops: Morning (a cafe/estate), Evening (dinner), Night (a local spot).
3. FORMAT: clean, with emojis.
"""

user_request = "Plan a 1-day trip to Coorg."
print(f"👤 USER (with persona): {user_request}\n" + "="*50)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"{wanderlust_persona}\nUSER REQUEST: {user_request}"
)
print(f"🤖 WANDERLUST AI:\n{response.text}")
# Same brain, same question - but now it has opinions.

👤 USER (with persona): Plan a 1-day trip to Coorg.
🤖 WANDERLUST AI:
Listen, if you came here for the crowded viewpoints and plastic trinkets, you’re in the wrong place. Put your phone away and actually taste the dirt and the beans. Here is how you do a day in Coorg without looking like a clueless tourist.

**Morning: Beyond the Beans** ☕
Skip the "coffee museum" scams. Head to **Ainmane Coffee** in Madikeri for the real deal. Don’t just sip; talk to the folks there about the nuances of Arabica vs. Robusta grown in the shade of the Western Ghats. Grab a bag of single-estate beans and leave. It’s about the soil, not the branding.

**Evening: The Ancestral Kitchen** 🍛
Go to **Coorg Cuisine** in Madikeri. Forget the generic "multi-cuisine" nonsense. You are here for *Pandi Curry* (pork with kachampuli—the souring agent is non-negotiable) and *Kadambuttu* (steamed rice dumplings). If the fat doesn't coat your throat, you aren't doing it right. Eat with your hands or don't bother.

**Night: 

## Snippet 3 - The Senses - it can see

_2 - The Senses_

The same brain that writes can also see. Point it at any photo (callback to your Day 9 Computer Vision session).

In [4]:
# The same brain that writes can also SEE. Point it at any photo.
import requests

url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
img_bytes = requests.get(url).content

print("🖼️  Showing Gemini a photo it has never seen...\n" + "="*50)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "You are a travel foodie. Look at this photo. Identify the dish, "
        "its likely origin, and name ONE destination famous for it.",
        types.Part(inline_data=types.Blob(mime_type="image/jpeg", data=img_bytes)),
    ],
)
print(f"🤖 WANDERLUST AI sees:\n{response.text}")
# Swap the URL for ANY image link - or upload your own photo to Colab.

🖼️  Showing Gemini a photo it has never seen...
🤖 WANDERLUST AI sees:
As a travel foodie, I’d recognize those beautiful, crumbly treats anywhere!

**The Dish:** Blueberry Scones.

**Likely Origin:** The United Kingdom (specifically Scotland). Scones evolved from a flat, round cake baked on a griddle, eventually becoming the quintessential component of a classic British afternoon tea.

**Famous Destination:** **Cornwall, England.** The region is legendary for its traditional cream teas, where warm, freshly baked scones are served with clotted cream and jam (the "Devonshire" vs. "Cornish" method of layering the toppings is a famous, long-standing culinary debate!).


## Snippet 4 - The Hands - reach the live web

_3 - The Hands_

Until now it only knew its training data. Flip ON Google Search and the brain breaks out of the jar - real, current info.

In [6]:
# Give the brain HANDS: the live web. Just switch the search tool ON.
search_tool = [types.Tool(google_search=types.GoogleSearch())]

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(tools=search_tool)
)

user_msg = ("What's the best way to travel from Coimbatore to Coorg this weekend, "
            "and how's the weather there right now?")
print(f"👤 USER: {user_msg}\n" + "="*50)
response = chat.send_message(user_msg)
print(f"🤖 WANDERLUST AI (live search):\n{response.text}")

👤 USER: What's the best way to travel from Coimbatore to Coorg this weekend, and how's the weather there right now?
🤖 WANDERLUST AI (live search):
For your trip from Coimbatore to Coorg this weekend (May 31st - June 1st, 2026), here's a breakdown of the best travel methods and the expected weather:

**Best Way to Travel from Coimbatore to Coorg**

The most convenient and generally preferred way to travel from Coimbatore to Coorg is **by road**, either by car/taxi or bus, due to the scenic routes and Coorg not having a direct train station or airport.

*   **By Car/Taxi (Recommended for flexibility and speed):**
    *   The road distance is approximately 280 to 325 kilometers, and the journey typically takes around 5 to 8 hours, including short breaks.
    *   This option offers scenic views and flexible travel plans.
    *   One-way cab rentals start from around ₹3,400.

*   **By Bus (Economical option):**
    *   Karnataka State Road Transport Corporation (KSRTC) operates buses, with 

**But did it really search, or hallucinate?** Let's check the *grounding metadata* - the actual sources it read.

In [7]:
# Proof it didn't make it up: the real web sources behind the answer.
from IPython.display import display, HTML

meta = response.candidates[0].grounding_metadata
if meta and meta.search_entry_point:
    display(HTML(meta.search_entry_point.rendered_content))
if meta and meta.grounding_chunks:
    print("\n🔗 Real sources it used:")
    for chunk in meta.grounding_chunks:
        if chunk.web:
            print(f" - {chunk.web.title}: {chunk.web.uri}")


🔗 Real sources it used:
 - pollachitours.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFCSxITroZjDkks3Z2dQcoTzbxhbR5Nn1mWt4hF5zuO4ev27IrCOR9eoxEvFSbMMTq1JEpuTCaNNC7F3KFyUw_NInRwdODG0EoznedQRE3JsJ911lrYWtrDdc-ee7maU9uf5Yb4Rp9iELXVFA==
 - rome2rio.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFJoZTzXVqMSvVZBScBqdKO3rqb0eUnSGBF-9hJZqei5O91-WsKio-hukSvZx7braejmWGcmOxOiv1SRozbHHzkcFChtMNvf14qY4SBAfr6x9Tvf7CCGyT0Oo9qvN5iIseaMGrsqpfERVYFSvdZpM3tPmdzJq4k4PM=
 - amy.cab: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEp7lrU5Uw1e_9r96zDwqqA4-_ioeyegeU5SJ_tjl9Gr3K4BkY6eZxAKRjU6wG9xL6bWkfugre21CdLQjNjDBgIdTljuMhreq5f3A9NLV94JdzsXZNSl0hwhggloMzIn5wKEe1XkzD4b2FT0DyNqneq1mA8FjzWqw-CRUOryPNHwCUi
 - clearcarrental.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEJsuNTowbeQEO7N6wUG6gmVwhzZ9EhJPcBD47hE_2OD-LzkbrB4jAV-G1OVHBWv7JZl-6NH-RAiq-13mzBnNw8OejgWQaPwsLaOII6Md2wTSLC3eI4ZH6vB1cBjD954AZLXaVekuozOF

## Snippet 5 - The Will - it runs YOUR code

_4 - The Will_

The deepest idea today: an LLM that decides to call your Python function. We build the loop BY HAND so you see every step - Think -> Act -> Observe. This is the moment an LLM becomes an agent.

In [8]:
# 1. THE TOOL - just a normal Python function with a clear docstring.
def estimate_trip_budget(nights: int, hotel_per_night: int, travel_cost: int):
    """Estimates the total cost of a trip in Indian Rupees.
    Args:
        nights: number of nights staying.
        hotel_per_night: hotel cost per night in INR.
        travel_cost: total round-trip travel cost in INR.
    """
    print(f"   ⚙️ [YOUR PYTHON RUNS] nights={nights}, hotel/night={hotel_per_night}, travel={travel_cost}")
    return {"total_inr": nights * hotel_per_night + travel_cost}

# 2. THE AGENT LOOP (we drive it ourselves to see inside)
def run_agent(user_query):
    print(f"👤 USER: {user_query}\n" + "="*60)
    history = [types.Content(role="user", parts=[types.Part(text=user_query)])]
    config = types.GenerateContentConfig(
        tools=[estimate_trip_budget],
        automatic_function_calling={"disable": True}   # manual loop
    )

    for turn in range(5):
        print(f"\n🔄 TURN {turn+1}")
        response = client.models.generate_content(model=MODEL_ID, contents=history, config=config)
        part = response.candidates[0].content.parts[0]

        # Did it just talk? Then we're done.
        if part.text and not part.function_call:
            print(f"🤖 AGENT: {part.text}")
            print("\n✅ DONE")
            break

        # Otherwise it wants to ACT - call a tool.
        if part.function_call:
            fc = part.function_call
            print(f"🧠 THOUGHT: I should call '{fc.name}' with {dict(fc.args)}")
            result = estimate_trip_budget(**fc.args)
            print(f"👀 OBSERVATION: tool returned {result}")
            history.append(response.candidates[0].content)
            history.append(types.Content(role="user", parts=[types.Part(
                function_response=types.FunctionResponse(name=fc.name, response=result)
            )]))

run_agent("Coorg for 2 nights, hotels about 3000 a night, travel 2500 round trip. "
          "What's my total, and is 10000 enough?")

👤 USER: Coorg for 2 nights, hotels about 3000 a night, travel 2500 round trip. What's my total, and is 10000 enough?

🔄 TURN 1
🧠 THOUGHT: I should call 'estimate_trip_budget' with {'nights': 2, 'hotel_per_night': 3000, 'travel_cost': 2500}
   ⚙️ [YOUR PYTHON RUNS] nights=2, hotel/night=3000, travel=2500
👀 OBSERVATION: tool returned {'total_inr': 8500}

🔄 TURN 2
🤖 AGENT: Your total for the accommodation and travel will be **₹8,500**.

Since your budget is **₹10,000**, you have **₹1,500 remaining**. 

While ₹8,500 covers the fixed costs, whether ₹10,000 is "enough" depends on your daily spending habits for food, local transport, and sightseeing activities once you are in Coorg. If you plan to eat at budget-friendly local restaurants and limit your sightseeing to a few key spots, ₹1,500 should be sufficient for your 2-day stay. However, if you plan on doing many paid activities or dining at upscale cafes, you might find it a bit tight.

✅ DONE


In [27]:
# List the first few available models to see our options
print("Available Models:")
for m in client.models.list():
    if "gemini" in m.name:
        print(f"- {m.name}")

Available Models:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.5-flash
- models/gemini-3.1-flash-tts-preview
- models/gemini-robotics-er-1.5-preview
- models/gemini-robotics-er-1.6-preview
- models/gemini-2.5-computer-use-preview-10-2025
- models/gemini-embedding-001
- models/gemini-

## Snippet 6 - The Team - agents working together

_5 - The Team (hero)_

Two specialist agents work AT THE SAME TIME, then a manager fuses their reports into one plan. This is real orchestration - and it's the moment to lean back and watch.

In [31]:
# THE HERO: two specialists in parallel streaming their thoughts, then a manager synthesis.
import asyncio
import ipywidgets as widgets
from IPython.display import display

async def run_worker(name, icon, prompt, out):
    with out:
        print(f"{icon} {name} initialized. Accessing live web tools...\n")
        print("-" * 45 + "\n")

    # Using generate_content_stream to stream responses chunk-by-chunk live
    response = await client.aio.models.generate_content_stream(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
    )

    full_text = ""
    async for chunk in response:
        if chunk.text:
            full_text += chunk.text
            with out:
                # print with flush=True to make the stream type out live in the widget box
                print(chunk.text, end="", flush=True)

    with out:
        print(f"\n\n✅ {icon} {name} complete.")
    return full_text

async def command_center(origin, destination, interest):
    print(f"👤 USER: Plan a trip from {origin} to {destination} ({interest}).\n")
    print("📣 MANAGER: dispatching two field agents in parallel streaming mode...\n")

    # Added height and overflow constraints to make the side-by-side layout clean and scrollable
    out_logi = widgets.Output(layout={"border": "1px solid #ccc", "width": "50%", "padding": "8px", "height": "350px", "overflow_y": "auto"})
    out_exp  = widgets.Output(layout={"border": "1px solid #ccc", "width": "50%", "padding": "8px", "height": "350px", "overflow_y": "auto"})
    display(widgets.HBox([out_logi, out_exp]))

    logistics, experience = await asyncio.gather(
        run_worker("LOGISTICS", "✈️", f"Best ways to travel from {origin} to {destination} this weekend, with rough costs and time.", out_logi),
        run_worker("EXPERIENCE", "🎒", f"Top {interest} experiences in {destination}, with specific names.", out_exp),
    )

    print("\n🧠 MANAGER: Field reports received. Synthesizing final itinerary...\n" + "="*55)
    final = await client.aio.models.generate_content(
        model=MODEL_ID,
        contents=f"""You are Wanderlust AI's trip manager. Fuse these two reports
into one tight, exciting 1-day plan with a short budget line.

[LOGISTICS]
{logistics}

[EXPERIENCE]
{experience}"""
    )
    print(f"🤖 FINAL ITINERARY:\n{final.text}")

await command_center("Coimbatore", "Coorg", "coffee estates & waterfalls")

👤 USER: Plan a trip from Coimbatore to Coorg (coffee estates & waterfalls).

📣 MANAGER: dispatching two field agents in parallel streaming mode...




🧠 MANAGER: Field reports received. Synthesizing final itinerary...
🤖 FINAL ITINERARY:
Hello! As your Wanderlust AI trip manager, I’ve distilled your weekend escape into a high-impact, 1-day itinerary. Pack your bags—Coorg awaits!

### **The "Coffee & Cascades" Day Trip**
*This plan maximizes your time by focusing on the Madikeri circuit, minimizing transit, and prioritizing the best of Coorg’s nature.*

*   **05:00 AM – Departure:** Depart Coimbatore via cab/self-drive (via NH 209). Early departure is key to beating traffic and reaching the coffee hills by 11:00 AM.
*   **11:30 AM – Morning Mist & Cascades:** Head straight to **Abbey Falls**. The short, scenic trek through spice plantations acts as the perfect wake-up call.
*   **01:00 PM – The "Bean-to-Cup" Experience:** Visit **Mercara Gold Estate**. Enjoy a guided tour of the plantation, learn the secrets of Arabica vs. Robusta, and finish with a fresh, estate-grown filter coffee in the café.
*   **03:00 PM – Hidden Gem:** Take a q

## Snippet 7 - The Framework - meet Google ADK

_6 - The Framework (teaser)_

You just ran the Think -> Act -> Observe loop by hand. Here's how the industry does it: with a framework, you DECLARE the agent and it runs the loop for you.

**Heads-up (the one heavy install today):** `google-adk` is large. If the import below errors, do **Runtime -> Restart session**, then re-run **Snippet 0** and this cell. (ADK runs via a `Runner` in code - the `adk web` / `adk run` commands are for a terminal, not a notebook cell.)

In [19]:
# The pro move: a FRAMEWORK runs the agent loop for you. You just DECLARE it.
!pip install -q google-adk

import os, uuid
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY   # ADK reads the key from here

from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gt

# DECLARE the agent - compare this to the by-hand loop you wrote in Snippet 5!
wanderlust = Agent(
    name="wanderlust_adk",
    model="gemini-2.5-flash",
    description="A travel buddy that uses live web search.",
    instruction="You are Wanderlust AI. Use Google Search for real, current info. Be concise and specific.",
    tools=[google_search],
)

session_service = InMemorySessionService()
runner = Runner(agent=wanderlust, app_name="wanderlust", session_service=session_service)

async def ask(query):
    sid = uuid.uuid4().hex
    await session_service.create_session(app_name="wanderlust", user_id="student", session_id=sid)
    msg = gt.Content(role="user", parts=[gt.Part(text=query)])
    async for event in runner.run_async(user_id="student", session_id=sid, new_message=msg):
        if event.is_final_response():
            print("🤖 Wanderlust (ADK):\n" + event.content.parts[0].text)

await ask("Plan a quick weekend trip from Coimbatore to Coorg with live travel options.")

🤖 Wanderlust (ADK):
Planning a quick weekend trip from Coimbatore to Coorg is feasible, with road travel being the most efficient option. The distance between Coimbatore and Coorg is approximately 280-325 km, and the journey typically takes around 5 to 8 hours by road, depending on the route and mode of transport.

Here’s a suggested itinerary and live travel options:

### **Weekend Itinerary: Coimbatore to Coorg**

**Travel Options & Duration:**
*   **By Car (Self-drive or Taxi):** This is the fastest and most flexible option, taking approximately 5-6 hours. A taxi can cost around ₹3,500 - ₹5,500 one way or about ₹9400 for a roundtrip.
*   **By Bus:** Direct buses are available from Coimbatore to Coorg (Madikeri). KSRTC (Karnataka State Road Transport Corporation) operates buses, with journey times ranging from 7 to 9 hours. Fares typically start from ₹382 to ₹1222. Some routes might involve transfers in Mysore.
*   **By Train:** There is no direct train to Coorg. The nearest major ra

## Snippet 8 - Bonus - why 'think step by step' works

_Bonus_

Fast answers vs. slow reasoning. Watch the model fix its own mistake when forced to show its work.

In [20]:
# 🔥 BONUS: fast (System 1) vs. step-by-step (System 2) thinking.
problem = ("Train: 2500. Hotel: 3000/night x 2 nights. 10% tax on hotel only. "
           "500 off the total. Is 9000 enough?")

fast = client.models.generate_content(model=MODEL_ID, contents=problem + " Answer Yes or No only.")
print("⚡ Fast answer:", fast.text)

slow = client.models.generate_content(model=MODEL_ID, contents=problem + " Let's think step by step.")
print("\n🐢 Step by step:\n", slow.text)

⚡ Fast answer: No.

🐢 Step by step:
 To determine if 9,000 is enough, let’s break down the costs step by step:

1.  **Train Cost:** 2,500
2.  **Hotel Base Cost:** 3,000 per night × 2 nights = 6,000
3.  **Hotel Tax:** 10% of the hotel base cost (6,000) = 600
4.  **Subtotal:** 2,500 (train) + 6,000 (hotel) + 600 (tax) = 9,100
5.  **Discount:** Subtract 500 from the total = 9,100 - 500 = 8,600

**Conclusion:** 
Your total cost is **8,600**. Since you have 9,000, it is enough, and you will have 400 remaining.


## Snippet 9 - Bonus - give your AI private memory (RAG)

_Bonus_

Embeddings (remember Day 6?) let your agent search private data it was never trained on. This is what they're FOR.

In [22]:
# 🔥 BONUS: Retrieval-Augmented Generation with embeddings.
import numpy as np

deals = [
    {"id": "D1", "text": "Coorg homestay inside a coffee estate. 2500/night. Breakfast included."},
    {"id": "D2", "text": "Ooty heritage hotel. 4000/night. Garden view."},
    {"id": "D3", "text": "Munnar tea-garden cottage. 3000/night. Misty hills."},
    {"id": "D4", "text": "Kodaikanal lakeside room. 2000/night. Budget pick."},
]

def embed(text):
    r = client.models.embed_content(model="gemini-embedding-2", contents=text)
    return np.array(r.embeddings[0].values)

for d in deals:
    d["vec"] = embed(d["text"])

def search(query):
    q = embed(query)
    return max(deals, key=lambda d: float(np.dot(d["vec"], q)))

query = "cheap place to stay near coffee plantations"
hit = search(query)
print(f"🔎 Best match for '{query}':\n   {hit['text']}\n")

answer = client.models.generate_content(
    model=MODEL_ID,
    contents=f"Using ONLY this info: {hit['text']}\nAnswer the user: {query}"
).text
print(f"🤖 {answer}")

🔎 Best match for 'cheap place to stay near coffee plantations':
   Coorg homestay inside a coffee estate. 2500/night. Breakfast included.

🤖 You can stay at a homestay located directly inside a coffee estate in Coorg for 2500/night, which includes breakfast.
